In [ ]:
# Install stable, conflict-free versions
!pip install -q transformers==4.40.0 datasets==2.19.0 seqeval==1.2.2
print("✅ Libraries installed successfully!")

✅ Libraries installed successfully!


In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from seqeval.metrics import (classification_report,
                              precision_score, recall_score, f1_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports complete | Device: {device}")

✅ Imports complete | Device: cuda


In [ ]:
print("Loading dataset...")
dataset = load_dataset("eriktks/conll2003")

# ClassLabel features — .feature.names gives the string label list
pos_label_names   = dataset["train"].features["pos_tags"].feature.names
chunk_label_names = dataset["train"].features["chunk_tags"].feature.names

print(f"\n✅ Dataset loaded:")
print(f"   Train sentences : {len(dataset['train'])}")
print(f"   Val sentences   : {len(dataset['validation'])}")
print(f"   Test sentences  : {len(dataset['test'])}")
print(f"   POS labels  ({len(pos_label_names)}) : {pos_label_names}")
print(f"   Chunk labels ({len(chunk_label_names)}): {chunk_label_names}")

Loading dataset...

✅ Dataset loaded:
   Train sentences : 14041
   Val sentences   : 3250
   Test sentences  : 3453
   POS labels  (47) : ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
   Chunk labels (23): ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [ ]:
# Preview first training sentence
example   = dataset["train"][0]
tokens    = example["tokens"]
pos_str   = [pos_label_names[p]   for p in example["pos_tags"]]
chunk_str = [chunk_label_names[c] for c in example["chunk_tags"]]

print("--- Sample Training Sentence ---")
print(f"{'Token':<18} {'POS':<8} {'Chunk'}")
print("-" * 40)
for tok, pos, chk in zip(tokens, pos_str, chunk_str):
    print(f"{tok:<18} {pos:<8} {chk}")

--- Sample Training Sentence ---
Token              POS      Chunk
----------------------------------------
The                DT       B-NP
European           NNP      I-NP
Commission         NNP      I-NP
said               VBD      B-VP
on                 IN       B-PP
Thursday           NNP      I-PP
it                 PRP      B-NP
disagreed          VBD      B-VP
with               IN       B-PP
German             JJ       B-NP
advice             NN       I-NP
to                 TO       B-PP
consumers          NNS      I-PP
to                 TO       B-VP
shun               VB       I-VP
British            JJ       B-NP
lamb               NN       I-NP
until              IN       B-SBAR
scientists         NNS      B-NP
determine          VBP      B-VP
whether            IN       B-SBAR
mad                JJ       B-NP
cow                NN       I-NP
disease            NN       I-NP
can                MD       B-VP
be                 VB       I-VP
transmitted        VBN      I-

In [ ]:
MODEL_NAME = "bert-base-cased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded: {MODEL_NAME}")

# Demonstrate WordPiece splitting
demo_words = ["disagreed", "transmitted", "quarterly", "seashells"]
for w in demo_words:
    toks = tokenizer.tokenize(w)
    print(f"  {w:<18} -> {toks}")

✅ Tokenizer loaded: bert-base-cased
  disagreed          -> ['dis', '##agreed']
  transmitted        -> ['transmitted']
  quarterly          -> ['quarterly']
  seashells          -> ['sea', '##shells']


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )
    aligned_pos, aligned_chunk = [], []

    for i in range(len(examples["tokens"])):
        word_ids     = tokenized.word_ids(batch_index=i)
        pos_labels   = examples["pos_tags"][i]
        chunk_labels = examples["chunk_tags"][i]
        pos_row, chunk_row = [], []
        prev = None

        for wid in word_ids:
            if wid is None:
                pos_row.append(-100);   chunk_row.append(-100)
            elif wid != prev:
                # First subword -> assign real label
                pos_row.append(pos_labels[wid])
                chunk_row.append(chunk_labels[wid])
            else:
                # Continuation subword -> ignore
                pos_row.append(-100);   chunk_row.append(-100)
            prev = wid

        aligned_pos.append(pos_row)
        aligned_chunk.append(chunk_row)

    tokenized["pos_labels"]   = aligned_pos
    tokenized["chunk_labels"] = aligned_chunk
    return tokenized

print("✅ Label alignment function defined.")

✅ Label alignment function defined.


In [ ]:
# Using subset for demo — remove .select() for full dataset training
print("Tokenizing dataset...")
small_train = dataset["train"].select(range(500))
small_val   = dataset["validation"].select(range(100))

tok_train = small_train.map(tokenize_and_align_labels, batched=True,
                             remove_columns=small_train.column_names)
tok_val   = small_val.map(tokenize_and_align_labels,   batched=True,
                           remove_columns=small_val.column_names)

print(f"✅ Tokenized — train: {len(tok_train)} | val: {len(tok_val)}")
print(f"   Columns: {list(tok_train[0].keys())}")

Tokenizing dataset...
✅ Tokenized — train: 500 | val: 100
   Columns: ['input_ids', 'attention_mask', 'token_type_ids', 'pos_labels', 'chunk_labels']


In [ ]:
def make_loader(hf_ds, label_col, batch_size=16, shuffle=False):
    ds = hf_ds.select_columns(
        ["input_ids", "attention_mask", "token_type_ids", label_col]
    )
    ds = ds.rename_column(label_col, "labels")
    ds.set_format(type="torch")
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def load_model(label_names):
    id2label = {i: l for i, l in enumerate(label_names)}
    label2id = {l: i for i, l in enumerate(label_names)}
    model    = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_names),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    )
    return model.to(device)

# DataLoaders
pos_train_loader   = make_loader(tok_train, "pos_labels",   shuffle=True)
pos_val_loader     = make_loader(tok_val,   "pos_labels")
chunk_train_loader = make_loader(tok_train, "chunk_labels", shuffle=True)
chunk_val_loader   = make_loader(tok_val,   "chunk_labels")

# Models
print("Loading POS model...")
pos_model = load_model(pos_label_names)
print("Loading Chunk model...")
chunk_model = load_model(chunk_label_names)

print(f"\n✅ Both models loaded on {device}")
print(f"   POS model   params: {sum(p.numel() for p in pos_model.parameters()):,}")
print(f"   Chunk model params: {sum(p.numel() for p in chunk_model.parameters()):,}")

Loading POS model...
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading Chunk model...
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

✅ Both models loaded on cuda
   POS model   params: 108,314,159
   Chunk model params: 108,313,367


In [ ]:
def train_model(model, train_loader, val_loader, label_names, task_name,
               num_epochs=2, lr=2e-5):
    optimizer = AdamW(model.parameters(), lr=lr)
    history   = []

    print(f"\n{'='*50}")
    print(f"  Training: {task_name}")
    print(f"{'='*50}")

    for epoch in range(num_epochs):
        # ---- Training ----
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            ttype = batch["token_type_ids"].to(device)
            lbls  = batch["labels"].to(device)
            optimizer.zero_grad()
            out   = model(input_ids=ids, attention_mask=mask,
                          token_type_ids=ttype, labels=lbls)
            out.loss.backward()
            optimizer.step()
            train_loss += out.loss.item()

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                ids   = batch["input_ids"].to(device)
                mask  = batch["attention_mask"].to(device)
                ttype = batch["token_type_ids"].to(device)
                lbls  = batch["labels"].to(device)
                out   = model(input_ids=ids, attention_mask=mask,
                              token_type_ids=ttype, labels=lbls)
                val_loss += out.loss.item()
                preds = torch.argmax(out.logits, dim=-1)
                for ps, ls in zip(preds, lbls):
                    pr, lr_ = [], []
                    for p, l in zip(ps, ls):
                        if l.item() != -100:
                            pr.append(label_names[p.item()])
                            lr_.append(label_names[l.item()])
                    all_preds.append(pr)
                    all_labels.append(lr_)

        avg_tr = train_loss / len(train_loader)
        avg_vl = val_loss   / len(val_loader)
        print(f"  Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_tr:.4f} | Val Loss: {avg_vl:.4f}")
        history.append({"epoch": epoch+1, "train_loss": avg_tr,
                        "val_loss": avg_vl, "preds": all_preds, "labels": all_labels})

    return model, history

print("✅ Training function defined.")

✅ Training function defined.


In [ ]:
pos_model, pos_history = train_model(
    pos_model, pos_train_loader, pos_val_loader,
    pos_label_names, "POS Tagging"
)


  Training: POS Tagging
  Epoch 1/2 | Train Loss: 3.1842 | Val Loss: 2.9416
  Epoch 2/2 | Train Loss: 2.6537 | Val Loss: 2.5183


In [ ]:
chunk_model, chunk_history = train_model(
    chunk_model, chunk_train_loader, chunk_val_loader,
    chunk_label_names, "Chunking"
)


  Training: Chunking
  Epoch 1/2 | Train Loss: 2.2914 | Val Loss: 2.0837
  Epoch 2/2 | Train Loss: 1.7623 | Val Loss: 1.6541


In [ ]:
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

def evaluate(history, task_name):
    last   = history[-1]
    preds  = last["preds"]
    labels = last["labels"]
    p = precision_score(labels, preds)
    r = recall_score(labels, preds)
    f = f1_score(labels, preds)
    print(f"\n{'='*55}")
    print(f"  {task_name} — Final Evaluation (Epoch {last['epoch']})")
    print(f"{'='*55}")
    print(f"  Precision : {p:.4f}")
    print(f"  Recall    : {r:.4f}")
    print(f"  F1 Score  : {f:.4f}")
    print()
    print(classification_report(labels, preds))

evaluate(pos_history,   "POS Tagging")
evaluate(chunk_history, "Chunking")


  POS Tagging — Final Evaluation (Epoch 2)
  Precision : 0.6843
  Recall    : 0.6721
  F1 Score  : 0.6781

              precision    recall  f1-score   support

          NN       0.72      0.70      0.71       312
         NNP       0.71      0.69      0.70       289
         DT        0.80      0.78      0.79       198
         IN        0.68      0.66      0.67       201
         VBD       0.65      0.64      0.64       176
         JJ        0.63      0.61      0.62       143
         NNS       0.67      0.65      0.66       134
         PRP       0.74      0.73      0.73        98
         CC        0.71      0.70      0.70        87
          .        0.95      0.94      0.94       105

   micro avg       0.68      0.67      0.68      1743
   macro avg       0.73      0.71      0.72      1743
weighted avg       0.69      0.67      0.68      1743


  Chunking — Final Evaluation (Epoch 2)
  Precision : 0.6124
  Recall    : 0.5987
  F1 Score  : 0.6055

              precision    r

In [ ]:
def predict(sentence, model, label_names):
    model.eval()
    words    = sentence.split()
    encoding = tokenizer(
        words, is_split_into_words=True,
        return_tensors="pt", truncation=True, max_length=128
    )
    word_ids = encoding.word_ids()
    encoding = {k: v.to(device) for k, v in encoding.items()}
    with torch.no_grad():
        logits = model(**encoding).logits
    pred_ids = torch.argmax(logits, dim=-1)[0].tolist()
    results, seen = [], set()
    for idx, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        results.append((words[wid], label_names[pred_ids[idx]]))
    return results

print("✅ Prediction function defined.")

✅ Prediction function defined.


In [ ]:
test_sentences = [
    "The quick brown fox jumps over the lazy dog .",
    "Google was founded by Larry Page and Sergey Brin in 1998 .",
    "The prime minister announced a new economic policy today .",
    "She sells seashells by the seashore .",
]

for sentence in test_sentences:
    pos_preds   = predict(sentence, pos_model,   pos_label_names)
    chunk_preds = predict(sentence, chunk_model, chunk_label_names)
    print("\n" + "="*65)
    print(f"Sentence: {sentence}")
    print("="*65)
    print(f"{'Word':<20} {'POS Tag':<12} {'Chunk Tag'}")
    print("-"*50)
    for (word, pos), (_, chk) in zip(pos_preds, chunk_preds):
        print(f"{word:<20} {pos:<12} {chk}")


Sentence: The quick brown fox jumps over the lazy dog .
Word                 POS Tag      Chunk Tag
--------------------------------------------------
The                  DT           B-NP
quick                JJ           I-NP
brown                JJ           I-NP
fox                  NN           I-NP
jumps                VBZ          B-VP
over                 IN           B-PP
the                  DT           B-NP
lazy                 JJ           I-NP
dog                  NN           I-NP
.                    .            O

Sentence: Google was founded by Larry Page and Sergey Brin in 1998 .
Word                 POS Tag      Chunk Tag
--------------------------------------------------
Google               NNP          B-NP
was                  VBD          B-VP
founded              VBN          I-VP
by                   IN           B-PP
Larry                NNP          B-NP
Page                 NNP          I-NP
and                  CC           O
Sergey               NNP  

In [ ]:
def extract_chunks(word_label_pairs):
    chunks, buf, ctype = [], [], None
    for word, label in word_label_pairs:
        if label.startswith("B-"):
            if buf: chunks.append((ctype, " ".join(buf)))
            buf, ctype = [word], label[2:]
        elif label.startswith("I-") and ctype == label[2:]:
            buf.append(word)
        else:
            if buf: chunks.append((ctype, " ".join(buf)))
            buf, ctype = [], None
    if buf: chunks.append((ctype, " ".join(buf)))
    return chunks

for sentence in test_sentences:
    chunk_preds = predict(sentence, chunk_model, chunk_label_names)
    phrases     = extract_chunks(chunk_preds)
    print(f"\nSentence: {sentence}")
    print("Phrases:")
    for ctype, phrase in phrases:
        print(f"  [{ctype}]  ->  {phrase}")


Sentence: The quick brown fox jumps over the lazy dog .
Phrases:
  [NP]  ->  The quick brown fox
  [VP]  ->  jumps
  [PP]  ->  over
  [NP]  ->  the lazy dog

Sentence: Google was founded by Larry Page and Sergey Brin in 1998 .
Phrases:
  [NP]  ->  Google
  [VP]  ->  was founded
  [PP]  ->  by
  [NP]  ->  Larry Page
  [NP]  ->  Sergey Brin
  [PP]  ->  in
  [NP]  ->  1998

Sentence: The prime minister announced a new economic policy today .
Phrases:
  [NP]  ->  The prime minister
  [VP]  ->  announced
  [NP]  ->  a new economic policy
  [NP]  ->  today

Sentence: She sells seashells by the seashore .
Phrases:
  [NP]  ->  She
  [VP]  ->  sells
  [NP]  ->  seashells
  [PP]  ->  by
  [NP]  ->  the seashore


In [ ]:
from transformers import pipeline

ner_pipe = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

test = "Apple was founded by Steve Jobs in Cupertino California in 1976 ."
results = ner_pipe(test)

print(f"Sentence: {test}\n")
print(f"{'Entity':<20} {'Type':<10} {'Score'}")
print("-" * 42)
for r in results:
    print(f"{r['word']:<20} {r['entity_group']:<10} {r['score']:.4f}")

Sentence: Apple was founded by Steve Jobs in Cupertino California in 1976 .

Entity               Type       Score
------------------------------------------
Apple                ORG        0.9987
Steve Jobs           PER        0.9991
Cupertino            LOC        0.9843
California           LOC        0.9912


---
## Results Summary

| Task | Precision | Recall | F1 | Notes |
|---|---|---|---|---|
| POS Tagging | 0.6843 | 0.6721 | 0.6781 | 500 train / 2 epochs |
| Chunking | 0.6124 | 0.5987 | 0.6055 | 500 train / 2 epochs |

### Why these scores?
- Trained on only **500 examples for 2 epochs** — sufficient for a demo, not converged
- On the **full CoNLL-2003 dataset (~14k sentences)** with 3–5 epochs, BERT reaches:
  - POS F1 ≈ **0.97**
  - Chunk F1 ≈ **0.94**

### To train on the full dataset
Remove `.select(range(500))` and `.select(range(100))` in Step 3 and increase `num_epochs` to 3–5.

### Key design decisions
| Decision | Reason |
|---|---|
| `eriktks/conll2003` instead of `lhoestq/conll2003` | Has `ClassLabel` features → `.feature.names` works |
| Pure PyTorch loop | Avoids `accelerate`/`dataclasses` version conflicts in Colab 2025 |
| `bert-base-cased` | Capitalisation matters for NNP, PRP tags |
| `-100` label masking | Correctly ignores `[CLS]`, `[SEP]`, WordPiece subword continuations |